# Inventory Movements Data Generator

Generates synthetic data for uc1_raw_inventory_movements.
Run cells in order.

In [0]:
import uuid
import random
from datetime import date, datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, LongType, BooleanType
)

# ── Configuration ──────────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG      = dbutils.widgets.get("catalog")
SCHEMA       = dbutils.widgets.get("schema")
TABLE_NAME   = "uc1_raw_inventory_movements"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

RANDOM_SEED  = 42
random.seed(RANDOM_SEED)

TODAY = date.today()

# ── Reference Data ─────────────────────────────────────────────────────────

STORES       = [f"ST{str(i).zfill(3)}" for i in range(1, 26)]
WAREHOUSES   = ["WH-NORTH", "WH-SOUTH", "WH-EAST", "WH-WEST"]
LOCATIONS    = [f"LOC-{str(i).zfill(3)}" for i in range(1, 101)]
NORMAL_SKUS  = [f"SKU-{str(i).zfill(5)}" for i in range(10000, 10500)]
NEW_SKUS     = [f"SKU-{str(i).zfill(5)}" for i in range(90000, 90300)]
CATEGORIES   = [
    "GROCERY", "DAIRY", "BAKERY", "PRODUCE",
    "FROZEN", "BEVERAGES", "HOUSEHOLD", "PERSONAL_CARE"
]
SUPPLIERS    = [f"SUP-{str(i).zfill(3)}" for i in range(1, 51)]

EVENT_TYPES    = ["RECEIPT", "SALE", "ADJUSTMENT", "TRANSFER", "WRITE_OFF"]
EVENT_WEIGHTS  = [0.25, 0.45, 0.15, 0.10, 0.05]

CDC_OPERATIONS = ["INSERT", "UPDATE", "DELETE"]
CDC_WEIGHTS    = [0.70, 0.25, 0.05]

# ── Volume Parameters ──────────────────────────────────────────────────────

DAILY_MIN = 1000
DAILY_MAX = 1500

# ── Helper Functions ───────────────────────────────────────────────────────

def random_event_timestamp(business_date: date) -> datetime:
    """
    Near real-time events arrive throughout the day and night.
    """
    hour   = random.randint(0, 23)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    return datetime(
        business_date.year,
        business_date.month,
        business_date.day,
        hour, minute, second
    )


def generate_quantity_movement(event_type: str) -> tuple:
    """
    Generates consistent quantity_before, quantity_change, quantity_after.
    Returns: (qty_before, qty_change, qty_after)
    """
    qty_before = random.randint(0, 5000)

    if event_type == "RECEIPT":
        qty_change = random.randint(10, 500)
    elif event_type == "SALE":
        qty_change = -random.randint(1, min(50, qty_before + 1))
    elif event_type == "ADJUSTMENT":
        qty_change = random.randint(-100, 100)
    elif event_type == "TRANSFER":
        qty_change = -random.randint(1, min(200, qty_before + 1))
    elif event_type == "WRITE_OFF":
        qty_change = -random.randint(1, min(100, qty_before + 1))
    else:
        qty_change = 0

    qty_after = max(0, qty_before + qty_change)
    return qty_before, qty_change, qty_after


def generate_movement(
    business_date: date,
    sku_pool: list = None,
    force_null_event_id: bool = False,
    force_malformed: bool = False,
    force_qty_inconsistency: bool = False,
    force_negative_qty_after: bool = False,
    force_null_unit_cost: bool = False,
    event_type_override: str = None
) -> dict:
    """
    Generates a single inventory movement event.
    """
    pool       = sku_pool or NORMAL_SKUS
    sku        = random.choice(pool)
    event_type = event_type_override or random.choices(
        EVENT_TYPES, weights=EVENT_WEIGHTS
    )[0]
    cdc_op     = random.choices(CDC_OPERATIONS, weights=CDC_WEIGHTS)[0]
    warehouse  = random.choice(WAREHOUSES)
    store      = random.choice(STORES) if event_type in ("SALE", "TRANSFER") else None

    qty_before, qty_change, qty_after = generate_quantity_movement(event_type)

    # Quantity inconsistency: make qty_after wrong
    if force_qty_inconsistency:
        qty_after = qty_after + random.randint(1, 50)

    # Negative quantity: override qty_after to be negative
    if force_negative_qty_after:
        qty_after = -random.randint(1, 100)

    unit_cost = (
        None
        if force_null_unit_cost
        else round(random.uniform(0.50, 500.00), 2)
    )

    total_cost = (
        None
        if unit_cost is None
        else round(abs(qty_change) * unit_cost, 2)
    )

    return {
        "event_id"               : None if force_null_event_id
                                   else str(uuid.uuid4()),
        "event_timestamp"        : random_event_timestamp(business_date),
        "event_type"             : event_type,
        "business_date"          : business_date,
        "warehouse_id"           : warehouse,
        "store_id"               : store,
        "location_code"          : random.choice(LOCATIONS),
        "sku"                    : sku,
        "product_id"             : sku.replace("SKU", "PRD"),
        "category_code"          : random.choice(CATEGORIES),
        "supplier_id"            : random.choice(SUPPLIERS),
        "quantity_change"        : qty_change,
        "quantity_before"        : qty_before,
        "quantity_after"         : qty_after,
        "unit_cost"              : unit_cost,
        "total_cost_impact"      : total_cost,
        "reference_document_id"  : str(uuid.uuid4()),
        "reference_document_type": random.choice(
                                       ["PO", "TRANSFER", "SALE", "MANUAL"]
                                   ),
        "cdc_operation"          : cdc_op,
        "cdc_source_system"      : "INVENTORY_MGMT_v2",
        "cdc_sequence_number"    : random.randint(1000000, 9999999),
        "ingestion_timestamp"    : datetime.now(),
        "_rescued_data"          : '{"error": "schema_mismatch"}'
                                   if force_malformed else None
    }


from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, LongType
)

# ── Explicit Schema for raw_inventory_movements ────────────────────────────

INV_SCHEMA = StructType([
    StructField("event_id",                StringType(),    True),
    StructField("event_timestamp",         TimestampType(), False),
    StructField("event_type",              StringType(),    True),
    StructField("business_date",           DateType(),      False),
    StructField("warehouse_id",            StringType(),    True),
    StructField("store_id",               StringType(),    True),
    StructField("location_code",           StringType(),    True),
    StructField("sku",                     StringType(),    True),
    StructField("product_id",              StringType(),    True),
    StructField("category_code",           StringType(),    True),
    StructField("supplier_id",             StringType(),    True),
    StructField("quantity_change",         IntegerType(),   True),
    StructField("quantity_before",         IntegerType(),   True),
    StructField("quantity_after",          IntegerType(),   True),
    StructField("unit_cost",               DoubleType(),    True),
    StructField("total_cost_impact",       DoubleType(),    True),
    StructField("reference_document_id",   StringType(),    True),
    StructField("reference_document_type", StringType(),    True),
    StructField("cdc_operation",           StringType(),    True),
    StructField("cdc_source_system",       StringType(),    True),
    StructField("cdc_sequence_number",     LongType(),      True),
    StructField("ingestion_timestamp",     TimestampType(), True),
    StructField("_rescued_data",           StringType(),    True),
])


def write_batch(records: list, description: str):
    """
    Writes inventory movement records using explicit schema.
    """
    column_order = [field.name for field in INV_SCHEMA.fields]

    rows = [
        tuple(r.get(col) for col in column_order)
        for r in records
    ]

    df = spark.createDataFrame(rows, schema=INV_SCHEMA)

    df.write \
      .mode("append") \
      .partitionBy("business_date") \
      .saveAsTable(TARGET_TABLE)

    print(f"  ✓ {description}")
    print(f"    Records written : {len(records)}")
    print(f"    Business date   : {records[0]['business_date']}")
    print(f"    Target table    : {TARGET_TABLE}")

print("Configuration loaded.")
print(f"Target table: {TARGET_TABLE}")

## Batch 1: 30 Days Clean Historical Data

In [0]:
print("Generating 30 days of clean historical inventory data...")

historical_dates = [
    TODAY + timedelta(days=offset)
    for offset in range(-37, -7)
]

total_records = 0

for business_date in historical_dates:
    daily_volume = random.randint(DAILY_MIN, DAILY_MAX)
    records = [
        generate_movement(business_date)
        for _ in range(daily_volume)
    ]
    write_batch(records, f"Historical clean: {business_date}")
    total_records += daily_volume

print(f"\nBatch 1 complete. Total records: {total_records}")
print("Expected agent result: All checks PASS")

## Batch 2: Clean Live Day

In [0]:
clean_live_date = TODAY + timedelta(days=-6)
daily_volume    = random.randint(DAILY_MIN, DAILY_MAX)

print(f"Generating clean live inventory batch: {clean_live_date}")

records = [
    generate_movement(clean_live_date)
    for _ in range(daily_volume)
]

write_batch(records, f"Clean live batch: {clean_live_date}")

print("\nBatch 2 complete.")
print("Expected agent result: All checks PASS")

## Batch 3: Malformed Records Spike

Injects 18% non-null _rescued_data records.
Simulates Autoloader rescuing records due to
source schema drift.

Expected agent result:
  MALFORMED_RECORDS → CRITICAL (exceeds 5% critical threshold)

In [0]:
malformed_date    = TODAY + timedelta(days=-5)
daily_volume      = random.randint(DAILY_MIN, DAILY_MAX)
malformed_rate    = 0.18

malformed_count   = int(daily_volume * malformed_rate)
normal_count      = daily_volume - malformed_count

print(f"Generating malformed records batch: {malformed_date}")
print(f"Total records:    {daily_volume}")
print(f"Malformed:        {malformed_count} ({malformed_rate*100:.0f}%)")

records = []

for _ in range(normal_count):
    records.append(generate_movement(malformed_date))

for _ in range(malformed_count):
    records.append(generate_movement(malformed_date, force_malformed=True))

random.shuffle(records)

write_batch(records, f"Malformed records batch: {malformed_date}")

print("\nBatch 3 complete.")
print("Expected agent result:")
print("  MALFORMED_RECORDS → CRITICAL")
print(f"  Observed rescue rate: {malformed_rate*100:.0f}%")
print(f"  Warning threshold: 1.0%")
print(f"  Critical threshold: 5.0%")

## Batch 4: Quantity Consistency Violation

Breaks rule: quantity_before + quantity_change = quantity_after
for 9% of TRANSFER events.

Expected agent result:
  CROSS_COLUMN_CONSISTENCY on quantity_after → WARNING

In [0]:
qty_date         = TODAY + timedelta(days=-4)
daily_volume     = random.randint(DAILY_MIN, DAILY_MAX)
violation_rate   = 0.09

# Only TRANSFER events are affected
transfer_count   = int(daily_volume * 0.10)   # ~10% are transfers normally
violation_count  = int(transfer_count * violation_rate * 10)
normal_count     = daily_volume - violation_count

print(f"Generating quantity consistency batch: {qty_date}")
print(f"Total records:    {daily_volume}")
print(f"Violations:       {violation_count}")

records = []

for _ in range(normal_count):
    records.append(generate_movement(qty_date))

for _ in range(violation_count):
    records.append(
        generate_movement(
            qty_date,
            event_type_override="TRANSFER",
            force_qty_inconsistency=True
        )
    )

random.shuffle(records)

write_batch(records, f"Quantity consistency batch: {qty_date}")

print("\nBatch 4 complete.")
print("Expected agent result:")
print("  CROSS_COLUMN_CONSISTENCY on quantity_after → WARNING")

## Batch 5: Cardinality Drift in SKU

Introduces 300 new SKUs on top of the normal 500.
Distinct SKU count increases from ~500 to ~800 (60% increase).

Expected agent result:
  CARDINALITY_DRIFT on sku → CRITICAL (exceeds 50% critical threshold)

In [0]:
cardinality_date  = TODAY + timedelta(days=-3)
daily_volume      = random.randint(DAILY_MIN, DAILY_MAX)

# 70% records use normal SKUs, 30% use new SKUs
new_sku_count  = int(daily_volume * 0.30)
normal_count   = daily_volume - new_sku_count

print(f"Generating cardinality drift batch: {cardinality_date}")
print(f"Total records:      {daily_volume}")
print(f"Normal SKU records: {normal_count}")
print(f"New SKU records:    {new_sku_count}")
print(f"Normal SKU pool:    {len(NORMAL_SKUS)} SKUs")
print(f"New SKU pool:       {len(NEW_SKUS)} SKUs")

records = []

for _ in range(normal_count):
    records.append(generate_movement(cardinality_date, sku_pool=NORMAL_SKUS))

for _ in range(new_sku_count):
    records.append(generate_movement(cardinality_date, sku_pool=NEW_SKUS))

random.shuffle(records)

write_batch(records, f"Cardinality drift batch: {cardinality_date}")

# Check distinct SKUs
distinct_normal = len(set(
    r["sku"] for r in records if r["sku"] in NORMAL_SKUS
))
distinct_new = len(set(
    r["sku"] for r in records if r["sku"] in NEW_SKUS
))

print(f"\nDistinct normal SKUs in batch: {distinct_normal}")
print(f"Distinct new SKUs in batch:    {distinct_new}")
print(f"Total distinct SKUs:           {distinct_normal + distinct_new}")

print("\nBatch 5 complete.")
print("Expected agent result:")
print("  CARDINALITY_DRIFT on sku → CRITICAL")
print(f"  Baseline distinct SKUs: ~500")
print(f"  Observed distinct SKUs: ~{distinct_normal + distinct_new}")
print(f"  Warning threshold: 20%   Critical threshold: 50%")

## Batch 6: Negative quantity_after

Introduces negative quantity_after in 4% of records.
Stock levels should never go below zero.

Expected agent result:
  NUMERIC_RANGE on quantity_after → CRITICAL
  (min_value configured as 0, any negative is a violation)

In [0]:
numeric_date     = TODAY + timedelta(days=-2)
daily_volume     = random.randint(DAILY_MIN, DAILY_MAX)
negative_rate    = 0.04

negative_count   = int(daily_volume * negative_rate)
normal_count     = daily_volume - negative_count

print(f"Generating numeric range batch: {numeric_date}")
print(f"Total records:          {daily_volume}")
print(f"Negative qty_after:     {negative_count} ({negative_rate*100:.0f}%)")

records = []

for _ in range(normal_count):
    records.append(generate_movement(numeric_date))

for _ in range(negative_count):
    records.append(
        generate_movement(
            numeric_date,
            force_negative_qty_after=True
        )
    )

random.shuffle(records)

write_batch(records, f"Numeric range batch: {numeric_date}")

print("\nBatch 6 complete.")
print("Expected agent result:")
print("  NUMERIC_RANGE on quantity_after → CRITICAL")
print(f"  Violation rate: {negative_rate*100:.0f}%")
print(f"  Warning threshold: 0.1%   Critical threshold: 1.0%")

## Batch 7: Combined Issues

Three simultaneous issues:
  1. Volume 45% above normal (spike)
  2. Malformed records at 3%
  3. event_id null rate at 2% (CRITICAL — should never be null)

Expected agent result:
  VOLUME_ANOMALY → WARNING (45% above baseline)
  MALFORMED_RECORDS → WARNING (3% rescue rate)
  NULL_SPIKE on event_id → CRITICAL
  Overall → CRITICAL with AI invocation triggered

In [0]:
combined_date      = TODAY + timedelta(days=-1)
spiked_volume      = int(1250 * 1.45)    # 45% above baseline mean of 1250
malformed_rate     = 0.03
null_event_id_rate = 0.02

malformed_count    = int(spiked_volume * malformed_rate)
null_event_count   = int(spiked_volume * null_event_id_rate)
normal_count       = spiked_volume - malformed_count - null_event_count

print(f"Generating combined issues batch: {combined_date}")
print(f"Baseline mean volume:  ~1,250")
print(f"Injected volume:       {spiked_volume} (~45% above baseline)")
print(f"Malformed records:     {malformed_count} ({malformed_rate*100:.0f}%)")
print(f"Null event_id:         {null_event_count} ({null_event_id_rate*100:.0f}%)")
print(f"Normal records:        {normal_count}")

records = []

for _ in range(normal_count):
    records.append(generate_movement(combined_date))

for _ in range(malformed_count):
    records.append(generate_movement(combined_date, force_malformed=True))

for _ in range(null_event_count):
    records.append(generate_movement(combined_date, force_null_event_id=True))

random.shuffle(records)

write_batch(records, f"Combined issues batch: {combined_date}")

print("\nBatch 7 complete.")
print("Expected agent result:")
print(f"  VOLUME_ANOMALY → WARNING ({spiked_volume} vs ~1,250 baseline)")
print(f"  MALFORMED_RECORDS → WARNING ({malformed_rate*100:.0f}% rescue rate)")
print(f"  NULL_SPIKE on event_id → CRITICAL ({null_event_id_rate*100:.0f}% null rate)")
print("  Overall → CRITICAL")
print("  AI invocation → TRIGGERED")

## Verification: Confirm All Batches Were Written Correctly

In [0]:
print("=" * 65)
print("INVENTORY MOVEMENTS — DATA GENERATION VERIFICATION")
print("=" * 65)

# Row count by date
print("\n1. Event count by business_date:\n")
spark.sql(f"""
    SELECT
        business_date,
        COUNT(*)                                                AS event_count,
        COUNT(DISTINCT sku)                                     AS distinct_skus,
        ROUND(SUM(CASE WHEN event_id IS NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)      AS pct_null_event_id,
        ROUND(SUM(CASE WHEN _rescued_data IS NOT NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)      AS pct_malformed
    FROM {TARGET_TABLE}
    GROUP BY business_date
    ORDER BY business_date
""").show(50, truncate=False)

# Quantity consistency check
print("\n2. Quantity consistency violations (batch 4):\n")
spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_records,
        SUM(CASE
                WHEN quantity_before + quantity_change != quantity_after
                THEN 1 ELSE 0
            END)                                                AS violations,
        ROUND(SUM(CASE
                      WHEN quantity_before + quantity_change != quantity_after
                      THEN 1 ELSE 0
                  END) / COUNT(*) * 100, 2)                    AS violation_pct
    FROM {TARGET_TABLE}
    WHERE business_date = '{TODAY + timedelta(days=-4)}'
""").show(truncate=False)

# Negative quantity check
print("\n3. Negative quantity_after records (batch 6):\n")
spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_records,
        SUM(CASE WHEN quantity_after < 0 THEN 1 ELSE 0 END)    AS negative_qty,
        ROUND(SUM(CASE WHEN quantity_after < 0
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)      AS pct_negative
    FROM {TARGET_TABLE}
    WHERE business_date = '{TODAY + timedelta(days=-2)}'
""").show(truncate=False)

# SKU cardinality check
print("\n4. SKU cardinality by date (watch for spike on batch 5):\n")
spark.sql(f"""
    SELECT
        business_date,
        COUNT(DISTINCT sku) AS distinct_skus
    FROM {TARGET_TABLE}
    GROUP BY business_date
    ORDER BY business_date
""").show(50, truncate=False)

# Overall summary
print("\n5. Overall data generation summary:\n")
spark.sql(f"""
    SELECT
        MIN(business_date)              AS earliest_date,
        MAX(business_date)              AS latest_date,
        COUNT(DISTINCT business_date)   AS distinct_dates,
        COUNT(*)                        AS total_records,
        ROUND(AVG(daily_count), 0)      AS avg_daily_events
    FROM (
        SELECT business_date, COUNT(*) AS daily_count
        FROM {TARGET_TABLE}
        GROUP BY business_date
    )
""").show(truncate=False)

print("Verification complete.")